# NB00 — acquisition and integrity

**Purpose:** download / register sources, verify hashes, write the availability report.
**Inputs:** public URLs + existing `depmap_data/` in the class repo.
**Outputs:** `data/raw/**`, `data/reference/v2_source_manifest.csv`
**Gate:** all required sources present and hash-verified (`missing count ≤ 0`).
**Runtime:** minutes if files already local; hours if fetching Wu/GTEx.
**Required:** metabric, tcga_brca, gtex_breast, omnipath, gdsc2, depmap, wu_scrna.
SCAN-B and CPTAC are *not* required here (they gate NB13 / NB08).


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe
try:
    import certifi, os
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
except Exception:
    pass

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = False
N_SAMPLES  = None   # full TCGA-BRCA; do not cap
N_SC_CELLS = 25_000  # Wu reference subsample if RAM is tight
N_PATIENTS = None
N_DRUGS    = None

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    if "sample_ids" not in kwargs:
        kwargs.setdefault("cohort", False)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config — no magic numbers below this cell
FETCH_CORE = True          # METABRIC, GDSC2, OmniPath PKN, cBioPortal TCGA
FETCH_HEAVY = False        # Wu scRNA (~2 GB), GTEx breast extract, ALMANAC
MANIFEST_PATH = REF / "v2_source_manifest.csv"
DEPMAP_SRC = REPO_ROOT / "depmap_data"

from io_data import pick_data_file, is_real_data_file
from datetime import date
from acquire import (
    REQUIRED_NB00, download_url, link_or_copy, load_manifest, missing_required,
    register_file, empty_manifest, sha256_file,
)
import pandas as pd

TODAY = date.today().isoformat()
print("FETCH_CORE", FETCH_CORE, "FETCH_HEAVY", FETCH_HEAVY)
print("SCAN-B request checklist:", REF / "SCANB_ACCESS.md")
print((REF / "SCANB_ACCESS.md").read_text().split("## Fallback")[0])


In [ ]:
# Load / compute: register everything we can find; optionally fetch core archives.
manifest = empty_manifest() if not MANIFEST_PATH.exists() else load_manifest(MANIFEST_PATH)

# --- DepMap: reuse class-repo files (do not re-download) ---
if DEPMAP_SRC.is_dir():
    any_depmap = False
    for f in sorted(DEPMAP_SRC.glob("*.csv")):
        dest = link_or_copy(f, RAW / "depmap" / f.name)
        key = "depmap" if f.name == "CRISPRGeneEffect.csv" else f"depmap_{f.stem}"
        required = f.name == "CRISPRGeneEffect.csv"
        manifest = register_file(
            manifest, dataset="DepMap", source_key=key, local_path=dest,
            source_organisation="Broad Institute",
            source_page="https://depmap.org",
            retrieval_date=TODAY, intended_role="CRISPR essentiality / expression",
            required_for_nb00=required, notes="symlinked from class-repo depmap_data/",
            v2_root=V2_ROOT, licence_or_access_note="DepMap public",
        )
        any_depmap = True
    print("registered DepMap files:", any_depmap)

URLS = {
    "metabric": (
        "https://cbioportal-datahub.s3.amazonaws.com/brca_metabric.tar.gz",
        RAW / "metabric" / "brca_metabric.tar.gz",
        "cBioPortal", "METABRIC expression/CNA/clinical/methylation",
    ),
    "tcga_brca": (
        "https://cbioportal-datahub.s3.amazonaws.com/brca_tcga_pan_can_atlas_2018.tar.gz",
        RAW / "tcga_brca" / "brca_tcga_pan_can_atlas_2018.tar.gz",
        "cBioPortal / TCGA", "TCGA-BRCA RNA + clinical (cBioPortal stand-in for GDC STAR if GDC client absent)",
    ),
    "gdsc2": (
        "https://cog.sanger.ac.uk/cancerrxgene/GDSC_release8.5/GDSC2_fitted_dose_response_27Oct23.xlsx",
        RAW / "gdsc2" / "GDSC2_fitted_dose_response_27Oct23.xlsx",
        "Sanger CancerRxGene", "GDSC2 dose-response for ODE fitting",
    ),
}

if FETCH_CORE:
    for key, (url, dest, org, role) in URLS.items():
        try:
            print("fetching", key, url)
            download_url(url, dest)
        except Exception as e:
            print("download failed", key, e)
        manifest = register_file(
            manifest, dataset=key, source_key=key, local_path=dest,
            source_organisation=org, source_page=url, retrieval_date=TODAY,
            intended_role=role, required_for_nb00=True, v2_root=V2_ROOT,
            licence_or_access_note="public",
        )

# OmniPath signed interactions among ODE nodes (small; always try)
omni_path = RAW / "omnipath" / "pkn_signed.parquet"
try:
    import pandas as pd
    from topology import DEFAULT_EDGES
    nodes = pd.read_csv(REF / "ode_nodes.csv")["gene"].tolist()
    pkn = pd.DataFrame(DEFAULT_EDGES, columns=["source", "target", "interaction"])
    try:
        from omnipath.interactions import AllInteractions
        raw = AllInteractions.get(genesymbols=True)
        sub = raw.loc[
            raw["source_genesymbol"].isin(nodes) & raw["target_genesymbol"].isin(nodes)
        ].copy()
        if "is_stimulation" in sub.columns:
            sub = sub[(sub.get("consensus_direction", 1) == 1)]
            sub["interaction"] = sub["is_stimulation"].map({True: 1, 1: 1}).fillna(-1)
            pkn = sub.rename(columns={"source_genesymbol": "source", "target_genesymbol": "target"})[
                ["source", "interaction", "target"]
            ]
        pkn = pkn.loc[:, ~pkn.columns.duplicated()]
        print("OmniPath rows:", len(pkn))
    except Exception as e:
        print("OmniPath client unavailable, using literature prior PKN:", e)
    omni_path.parent.mkdir(parents=True, exist_ok=True)
    pkn.to_parquet(omni_path, index=False)
except Exception as e:
    print("PKN persist failed", e)

manifest = register_file(
    manifest, dataset="OmniPath", source_key="omnipath", local_path=omni_path,
    source_organisation="Saez-Rodriguez lab", source_page="https://omnipathdb.org",
    retrieval_date=TODAY, intended_role="signed PKN", required_for_nb00=True,
    v2_root=V2_ROOT, licence_or_access_note="OmniPath licence",
)

# Placeholders for heavy / controlled sources so the schema is complete
for key, page, role, required in [
    ("gtex_breast", "https://gtexportal.org", "GTEx v8 breast normal reference", True),
    ("wu_scrna", "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE176078", "Wu et al. scRNA BayesPrism reference", True),
    ("almanac", "https://wiki.nci.nih.gov/display/NCIDTPdata/NCI-ALMANAC", "NCI-ALMANAC ComboScores", False),
    ("cptac_brca", "https://proteomics.cancer.gov/data-portal", "CPTAC-BRCA proteomics (NB08)", False),
    ("scanb", "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE96058", "SCAN-B outcomes (NB13)", False),
]:
    dest = RAW / key / "PLACEHOLDER.txt"
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        dest.write_text(f"Put {key} files in this directory. See {page}\n")
    use = pick_data_file(dest.parent)
    if use is None:
        use = dest
        real = []
    else:
        real = [use]
    manifest = register_file(
        manifest, dataset=key, source_key=key, local_path=use,
        source_page=page, retrieval_date=TODAY, intended_role=role,
        required_for_nb00=required, v2_root=V2_ROOT,
        notes="placeholder until a real file is dropped in this folder" if not real else "found local file",
        licence_or_access_note="see source page",
    )
    if real:
        print("found local", key, real[0].name)
    else:
        print("MISSING", key, "— drop files in", dest.parent)


In [ ]:
# Persist manifest
manifest = manifest.drop_duplicates("source_key", keep="last")
manifest.to_csv(MANIFEST_PATH, index=False)
print(manifest[["source_key", "verified", "required_for_nb00", "local_path"]].to_string(index=False))


In [ ]:
# GATE
missing = missing_required(manifest)
ok = gate("NB00", "sources_available", len(missing), 0, direction="lte",
          n=len(REQUIRED_NB00),
          note=("missing: " + ", ".join(missing)) if missing else "all required sources present")
print("SCAN-B is NOT in required — submit the request in data/reference/SCANB_ACCESS.md")
assert isinstance(ok, bool)


In [ ]:
# A1 — PK table pair coverage (n_pairs, not drug count)
from pk_table import (
    count_almanac_pairs_fully_covered, coverage_note, load_almanac_named_pairs, load_pk_table,
)
pk_path = REF / "drug_pk.csv"
n_pairs = 0
note = "drug_pk.csv missing"
if pk_path.exists():
    pk = load_pk_table(pk_path)
    pairs = load_almanac_named_pairs(RAW / "almanac", REF)
    n_pairs = count_almanac_pairs_fully_covered(pk, pairs)
    note = coverage_note(pk, n_pairs)
gate("A1", "pk_table_coverage", float(n_pairs), 100, n=n_pairs, min_n=1, smoke_test=False, note=note)


In [ ]:
# Figures / availability bar
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
keys = list(REQUIRED_NB00)
vals = [int(bool(manifest.loc[manifest.source_key == k, "verified"].any())) if k in set(manifest.source_key) else 0 for k in keys]
ax.bar(keys, vals, color=["#2ca02c" if v else "#d62728" for v in vals])
ax.set_ylim(0, 1.2); ax.set_ylabel("verified"); ax.set_title("NB00 required sources")
plt.xticks(rotation=30, ha="right"); fig.tight_layout()
fig.savefig(FIGURES / "NB00_source_availability.png", dpi=140)
print("saved", FIGURES / "NB00_source_availability.png")
